In [1]:
import numpy as np
import pandas as pd
import wfdb
import os
import sys
from scipy import signal
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt
import pywt
import glob
import json
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
import datetime
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import train_test_split

In [2]:
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torch import nn
from torch import optim
import torch.nn.functional as F
from torchsummary import summary

In [17]:
class SignalCNN(nn.Module):
    def __init__(self, signal_length=1536, num_classes=9):
        super(SignalCNN, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=1),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=1),
            

        )
        
        self.classifier = nn.Sequential(
            nn.LazyLinear(512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.LazyLinear(num_classes)
        )
        
    def forward(self, x):
        # Add channel dimension if not present
        if len(x.shape) == 3:
            x = x.unsqueeze(1)
        # print(x.shape)
        x = self.features(x)
        # print(F.adaptive_avg_pool2d(x, (1, x.size()[2])).shape)
        x = F.adaptive_avg_pool2d(x, (1, x.size()[2]))
        
        x = x.view(x.size(0), -1)
        # print('x: ', x.size())
        # print(x.shape)
        x = self.classifier(x)
        return x

In [4]:
train_X = np.load(r'F:\EAGLe_explanation_attention_guided_learning_for_time_series\data\processed\CPSP2018\train_X.npy')
test_X = np.load(r'F:\EAGLe_explanation_attention_guided_learning_for_time_series\data\processed\CPSP2018\test_X.npy')
train_y = np.load(r'F:\EAGLe_explanation_attention_guided_learning_for_time_series\data\processed\CPSP2018\train_y.npy')
test_y = np.load(r'F:\EAGLe_explanation_attention_guided_learning_for_time_series\data\processed\CPSP2018\test_y.npy')

In [ ]:
train_X.shape, test_X.shape, train_y.shape, test_y.shape

In [6]:
class SignalDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.transform = transform
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        signal = self.X[idx]
        label = self.y[idx]

        if self.transform:
            signal = self.transform(signal)
            
        return signal, label

def create_signal_dataloader(X, y, batch_size=32, num_workers=4, shuffle=True, transform=None):
    dataset = SignalDataset(X, y, transform=transform)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        # num_workers=num_workers,
        pin_memory=True
    )

train_dl = create_signal_dataloader(
    X=train_X,
    y=train_y,
    batch_size=32,
)

test_dl = create_signal_dataloader(
    X=test_X,
    y=test_y,
    batch_size=32,
)

In [ ]:
from sklearn.metrics import f1_score
def train_model(model, train_loader, val_loader, id=0, learning_rate=0.001, num_epochs=50, device='cuda'):
    """
    Train the CNN model.
    
    Args:
        model: The CNN model
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        num_epochs: Number of training epochs
        device: Device to train on ('cuda' or 'cpu')
    
    Returns:
        model: Trained model
        history: Dictionary containing training history
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.9, patience=5)
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    best_val_loss = float('inf')
    best_val_acc = 0
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for signals, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            signals, labels = signals.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(signals)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * correct / total
        
        # Validation phase
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        all_labels = []
        all_predictions = []

        with torch.no_grad():
            for signals, labels in val_loader:
                signals, labels = signals.to(device), labels.to(device)
                outputs = model(signals)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                all_labels.append(labels.cpu())
                all_predictions.append(predicted.cpu())
        
        all_labels = torch.cat(all_labels).numpy()
        all_predictions = torch.cat(all_predictions).numpy()
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * (all_predictions == all_labels).sum() / all_labels.size

        # Update learning rate
        # scheduler.step(val_loss)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'best_model_{id}.pth')
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        print(classification_report(all_labels, all_predictions, digits=4))
        print('-' * 50)
    
    return model, history, best_val_acc
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [23]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class CustomResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=9):
        super(CustomResNet, self).__init__()
        self.in_channels = 64

        # Initial convolution layer adjusted for the input size
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # Adaptive pooling to handle variable input sizes
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        x = x.unsqueeze(1)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

def create_resnet18(num_classes=9):
    return CustomResNet(BasicBlock, [2, 2, 2, 2], num_classes)

In [ ]:
best_acc = 0
best_model = tuple
for i in range(2):
    # model = SignalCNN(signal_length=1536, num_classes=9)
    model = create_resnet18(num_classes=9)
    model, history, best_val_acc = train_model(model, train_dl, test_dl, id=i, learning_rate=0.0003, num_epochs=100, device=device)
    if best_acc < best_val_acc:
        best_acc = best_val_acc
        best_model = (model, history)

In [ ]:
def get_predictions(
    model: torch.nn.Module,
    test_loader: torch.utils.data.DataLoader,
    device: str = 'cuda'
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Get predictions from the model.
    
    Args:
        model: Trained PyTorch model
        test_loader: DataLoader containing test data
        device: Device to run predictions on
    
    Returns:
        Tuple of (true labels, predicted labels)
    """
    model.eval()
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for signals, labels in test_loader:
            signals = signals.to(device)
            outputs = model(signals)
            _, predicted = torch.max(outputs.data, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    
    return np.array(y_true), np.array(y_pred)

def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: Optional[List[str]] = None
) -> None:
    """
    Plot confusion matrix using seaborn.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        class_names: List of class names
    """
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    
    # Create normalized confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt='.2%',
        cmap='Blues',
        xticklabels=class_names if class_names else 'auto',
        yticklabels=class_names if class_names else 'auto'
    )
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def evaluate_model(
    model: torch.nn.Module,
    test_loader: torch.utils.data.DataLoader,
    class_names: Optional[List[str]] = None,
    device: str = 'cuda'
) -> None:
    """
    Evaluate model and print classification report with confusion matrix.
    
    Args:
        model: Trained PyTorch model
        test_loader: DataLoader containing test data
        class_names: List of class names
        device: Device to run evaluation on
    """
    # Get predictions
    y_true, y_pred = get_predictions(model, test_loader, device)
    
    # Print classification report
    print("\nClassification Report:")
    print("-" * 60)
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names if class_names else None,
        digits=4
    )
    print(report)
    
    # Calculate and print ROC AUC for binary classification
    if len(np.unique(y_true)) == 2:
        try:
            roc_auc = roc_auc_score(y_true, y_pred)
            print(f"\nROC AUC Score: {roc_auc:.4f}")
        except Exception as e:
            print(f"Could not calculate ROC AUC: {e}")
    
    # Print confusion matrix
    print("\nConfusion Matrix:")
    print("-" * 60)
    plot_confusion_matrix(y_true, y_pred, class_names)

In [ ]:
%matplotlib qt5
# Assuming you have your model and test_loader ready
class_names = ["Normal", "S", "V", "F"]  # Your class names

test_model = SignalCNN(signal_length=300, num_classes=4).to(device)
state_dict = torch.load('best_model_0.pth')
test_model.load_state_dict(state_dict)

# test_model = best_model[0]

# Evaluate the model
evaluate_model(
    model=test_model,
    test_loader=test_dl,
    class_names=class_names,
    device='cuda'  # or 'cpu' if you're not using GPU
)